In [31]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression

Decide which features have normal distribution and then use proper test for statistical significance

In [42]:
# Load all merged spine datasets
df_l_c25 = pd.read_csv(r"D:\DP\CSV\merged_CaSupp_25_radiomics_spine_lesions_features.csv")
# df_l_konv = pd.read_csv(r"D:\DP\CSV\merged_konv_radiomics_spine_lesions_features.csv")
# df_l_vmi40 = pd.read_csv(r"D:\DP\CSV\merged_monoe_40kev_radiomics_spine_lesions_features.csv")
# df_l_vmi80 = pd.read_csv(r"D:\DP\CSV\merged_monoe_80kev_radiomics_spine_lesions_features.csv")
# df_l_vmi120 = pd.read_csv(r"D:\DP\CSV\merged_monoe_120kev_radiomics_spine_lesions_features.csv")

# Load clinical data
df_clinical = pd.read_csv(r"D:\DP\Table_clinical_data.csv", encoding="cp1252")

NORMALITY DISTRIBUTION TEST

p > 0.05 -> normal distribution || p ≤ 0.05 -> not normal distribution

In [43]:
# Divide features based on their normality distribution (normal/non-normal)
def normality_distribution(dataset):
    # Keep only numeric columns
    numeric_cols = dataset.select_dtypes(include=['number']).columns
    dataset_clean = dataset[numeric_cols]

    # Run Shapiro test
    normal_distribution = []
    non_normal_distribution = []

    for column in numeric_cols:
        # Shapiro fails if less than 3 samples → skip such columns
        if dataset_clean[column].shape[0] < 3:
            continue

        _, p_val = shapiro(dataset_clean[column])
        if p_val > 0.05:
            normal_distribution.append(column)
        else:
            non_normal_distribution.append(column)

    # Return results
    return normal_distribution, non_normal_distribution, dataset_clean


normal_features, non_normal_features, features_df = normality_distribution(df_l_c25)

print(f"Number of normal features: {len(normal_features)}")
print(f"Number of non-normal features: {len(non_normal_features)}")

Number of normal features: 71
Number of non-normal features: 130


In [44]:
# Divide clinical data based on their normality distribution
normal_clinical, non_normal_clinical, clinical_df = normality_distribution(df_clinical)

print(f"Number of normal clinical: {len(normal_clinical)}")
print(f"Number of non-normal clinical: {len(non_normal_clinical)}")

Number of normal clinical: 0
Number of non-normal clinical: 10


In [35]:
# Standardized features and also clinical data
scaler = StandardScaler()

# scale each dataset separately
features_scaled = pd.DataFrame(scaler.fit_transform(features_df), columns=features_df.columns)
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df), columns=clinical_df.columns)

In [29]:
corr_matrix = pd.DataFrame(np.zeros((len(features_df.columns), len(clinical_df.columns))), index=features_df.columns, columns=clinical_df.columns)

for f_col in features_df.columns:
    for c_col in clinical_df.columns:
        x = features_df[f_col]
        y = clinical_df[c_col]

        mask = x.notna() & y.notna()
        if mask.sum() > 1:
            r, _ = spearmanr(x[mask], y[mask])
            corr_matrix.loc[f_col, c_col] = r
        else:
            corr_matrix.loc[f_col, c_col] = np.nan

In [30]:
corr_long = corr_matrix.reset_index().melt(
    id_vars='index',
    var_name='Clinical',
    value_name='Correlation'
).rename(columns={'index': 'Feature'})

# drop NaNs just in case
corr_long = corr_long.dropna()

# sort by correlation value
corr_sorted = corr_long.sort_values('Correlation', ascending=False)

# top 10 positive correlations
top_pos = corr_sorted.head(10)

# top 10 negative correlations
top_neg = corr_sorted.tail(10)

print("🔹 Top 10 positive correlations:")
print(top_pos.to_string(index=False))

print("\n🔸 Top 10 negative correlations:")
print(top_neg.to_string(index=False))

🔹 Top 10 positive correlations:
                                          Feature                   Clinical  Correlation
       gradient_glrlm_LongRunLowGrayLevelEmphasis Beta2 microglobulin (mg/l)     0.484563
            gradient_gldm_LargeDependenceEmphasis Beta2 microglobulin (mg/l)     0.475191
gradient_gldm_LargeDependenceLowGrayLevelEmphasis Beta2 microglobulin (mg/l)     0.473620
                       original_glrlm_RunVariance Beta2 microglobulin (mg/l)     0.468230
                       gradient_glrlm_RunVariance Beta2 microglobulin (mg/l)     0.467902
            original_gldm_LargeDependenceEmphasis Beta2 microglobulin (mg/l)     0.467408
                   original_glrlm_LongRunEmphasis Beta2 microglobulin (mg/l)     0.464906
               gradient_gldm_LowGrayLevelEmphasis Beta2 microglobulin (mg/l)     0.464723
                   gradient_glrlm_LongRunEmphasis Beta2 microglobulin (mg/l)     0.462859
                 gradient_gldm_DependenceVariance Beta2 microglobuli

Try PLS-Partial Least Squares and CCA-Canonical Correlation Analysis

In [45]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
clinical_df = pd.DataFrame(imputer.fit_transform(clinical_df), columns=clinical_df.columns)

# Standardized features and also clinical data
scaler = StandardScaler()

# scale each dataset separately
features_scaled = pd.DataFrame(scaler.fit_transform(features_df), columns=features_df.columns)
clinical_scaled = pd.DataFrame(scaler.fit_transform(clinical_df), columns=clinical_df.columns)

# Initialize PLS n_components = number of latent variables to keep (tune this)
pls = PLSRegression(n_components=2)

# Fit model
pls.fit(features_scaled, clinical_scaled)

# Transform X and Y to latent components
X_scores, Y_scores = pls.transform(features_scaled, clinical_scaled)

In [52]:
coef_df = pd.DataFrame(pls.coef_.T, index=features_df.columns, columns=clinical_df.columns)

top_features_dict = {}

for col in coef_df.columns:
    # Sort features by absolute coefficient
    top_features = coef_df[col].abs().sort_values(ascending=False).head(10)
    top_features_dict[col] = top_features

for k, v in top_features_dict.items():
    print(f"{k}: {v}")
    print()

Serum M-protein quantity (g/l): gradient_glrlm_RunEntropy                          0.007553
gradient_glszm_HighGrayLevelZoneEmphasis           0.006841
gradient_gldm_DependenceEntropy                    0.006674
gradient_glszm_GrayLevelNonUniformityNormalized    0.006629
gradient_glcm_Imc2                                 0.006497
gradient_glrlm_GrayLevelVariance                   0.006432
gradient_glszm_LowGrayLevelZoneEmphasis            0.006382
gradient_glszm_SmallAreaHighGrayLevelEmphasis      0.006315
gradient_glrlm_LongRunHighGrayLevelEmphasis        0.006193
gradient_gldm_GrayLevelVariance                    0.006051
Name: Serum M-protein quantity (g/l), dtype: float64

Serum kappa FLC quantity (mg/l: gradient_gldm_DependenceVariance                     0.004391
gradient_gldm_LargeDependenceLowGrayLevelEmphasis    0.004016
original_glrlm_RunVariance                           0.003819
original_gldm_DependenceVariance                     0.003795
gradient_glrlm_LongRunLowGrayLevel